<a href="https://colab.research.google.com/github/Capaquira/projectsLivreLow-code/blob/Notebooks/EnhancedML_keras_ipynb1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

!wget -q https://storage.googleapis.com/low-code-ai-book/car_prices_train.csv
!wget -q https://storage.googleapis.com/low-code-ai-book/car_prices_valid.csv
!wget -q https://storage.googleapis.com/low-code-ai-book/car_prices_test.csv

train_df = pd.read_csv('./car_prices_train.csv')
y_train = train_df['sellingprice']
X_train = train_df.drop('sellingprice', axis=1)

valid_df = pd.read_csv('./car_prices_valid.csv')
y_valid = valid_df['sellingprice']
X_valid = valid_df.drop('sellingprice', axis=1)


In [2]:
import tensorflow as tf
from tensorflow.keras.layers import StringLookup, HashedCrossing, Discretization, Concatenate

cat_cols = ['make', 'model', 'trim', 'body', 'transmission', 'state',
            'color', 'interior']
num_cols = ['odometer', 'year', 'condition']

inputs = {}

for col in cat_cols:
  inputs[col] = tf.keras.Input(shape=(1,), name=col,
                               dtype = tf.string)

for col in num_cols:
  inputs[col] = tf.keras.Input(shape=(1,), name=col, dtype = tf.int64)

In [3]:
preproc_layers = {}
for col in cat_cols:
  layer = StringLookup(output_mode='one_hot')
  layer.adapt(X_train[col])
  preproc_layers[col] = layer(inputs[col])


In [4]:
for col in num_cols:
  layer = Discretization(num_bins=10,
                         output_mode='one_hot')
  layer.adapt(X_train[col])
  preproc_layers[col] = layer(inputs[col])


In [5]:
model_trim = tf.keras.layers.HashedCrossing(num_bins=1000, output_mode='one_hot')((inputs['model'], inputs['trim']))
color_int = tf.keras.layers.HashedCrossing(num_bins=400, output_mode='one_hot')((inputs['color'], inputs['interior']))

preproc_layers['model_trim'] = model_trim
preproc_layers['color_int'] = color_int
